In [21]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

catalog_path = "/Users/nityaarya/Downloads/Worth-the-Watch/Data/Processed/combined_streaming_catalog_with_originals.csv"
catalog_df = pd.read_csv(catalog_path)

# Load pricing data 
pricing_data = {
    'Platform': ['Netflix', 'Amazon Prime Video', 'Hulu', 'Max', 'Apple TV+'],
    'Plan': ['Standard (Ad-Free)', 'Prime with Video', 'No Ads', 'Ad-Free', 'Standard'],
    'Monthly Price': [17.99, 17.99, 18.99, 16.99, 9.99]
}
pricing_df = pd.DataFrame(pricing_data)

# Step 1: Platform-Level Summary Table
summary = catalog_df.groupby('platform').agg(
    Total_Titles=('title', 'count'),
    Total_Originals=('is_original', lambda x: (x == 'Y').sum()),
    Avg_IMDb_Rating=('imdbAverageRating', 'mean'),
    Weighted_IMDb_Rating=('imdbAverageRating', lambda x: np.average(
        x.fillna(0), weights=catalog_df.loc[x.index, 'imdbNumVotes'].fillna(0))),
    Genre_Variety=('genres', lambda x: len(set(g for sublist in x.dropna().apply(eval) for g in sublist)))
).reset_index()

# Step 2: Map platform names to match pricing data
platform_name_map = {
    'netflix': 'Netflix',
    'prime': 'Amazon Prime Video',
    'hulu': 'Hulu',
    'hbo_max': 'Max',
    'apple_tv': 'Apple TV+'
}
summary['Platform'] = summary['platform'].map(platform_name_map)

summary = summary.merge(pricing_df, on='Platform', how='left')

summary['Titles_per_Dollar'] = summary['Total_Titles'] / summary['Monthly Price']
summary['Originals_per_Dollar'] = summary['Total_Originals'] / summary['Monthly Price']
summary['Weighted_IMDb_per_Dollar'] = summary['Weighted_IMDb_Rating'] / summary['Monthly Price']

scaler = MinMaxScaler()
summary[['Norm_Titles_per_Dollar', 'Norm_Weighted_IMDb_per_Dollar', 'Norm_Originals_per_Dollar']] = scaler.fit_transform(
    summary[['Titles_per_Dollar', 'Weighted_IMDb_per_Dollar', 'Originals_per_Dollar']]
)

# Calculate composite Value Index
summary['Value_Index'] = (
    summary['Norm_Titles_per_Dollar'] +
    summary['Norm_Weighted_IMDb_per_Dollar'] +
    summary['Norm_Originals_per_Dollar']
)

summary_sorted = summary.sort_values(by='Value_Index', ascending=False)

summary_sorted[['Platform', 'Monthly Price', 'Total_Titles', 'Total_Originals',
                'Avg_IMDb_Rating', 'Weighted_IMDb_Rating', 'Genre_Variety',
                'Titles_per_Dollar', 'Originals_per_Dollar', 'Weighted_IMDb_per_Dollar',
                'Value_Index']]


,Platform,Monthly Price,Total_Titles,Total_Originals,Avg_IMDb_Rating,Weighted_IMDb_Rating,Genre_Variety,Titles_per_Dollar,Originals_per_Dollar,Weighted_IMDb_per_Dollar,Value_Index
0,Apple TV+,9.99,16928,208,6.367966,7.200682,31,1694.494494,20.820821,0.720789,1.472418
3,Netflix,17.99,20084,1967,6.400291,7.258466,29,1116.397999,109.338521,0.403472,1.223061
4,Amazon Prime Video,17.99,65850,443,5.952789,7.171655,34,3660.366870,24.624792,0.398647,1.142224
1,Max,16.99,9166,960,6.693769,7.388401,31,539.493820,56.503826,0.434868,0.595843
2,Hulu,18.99,9308,224,6.574679,7.504130,31,490.152712,11.795682,0.395162,0.000000


In [26]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import plotly.express as px
import plotly.graph_objects as go
from dash import Dash, html, dcc

# Pricing data manually entered
pricing_data = {
    'Platform': ['Netflix', 'Amazon Prime Video', 'Hulu', 'Max', 'Apple TV+'],
    'Plan': ['Standard (Ad-Free)', 'Prime with Video', 'No Ads', 'Ad-Free', 'Standard'],
    'Monthly Price': [17.99, 17.99, 18.99, 16.99, 9.99]
}
pricing_df = pd.DataFrame(pricing_data)

# Platform mapping
platform_name_map = {
    'netflix': 'Netflix',
    'prime': 'Amazon Prime Video',
    'hulu': 'Hulu',
    'hbo_max': 'Max',
    'apple_tv': 'Apple TV+'
}

# Summary table
summary = catalog_df.groupby('platform').agg(
    Total_Titles=('title', 'count'),
    Total_Originals=('is_original', lambda x: (x == 'Y').sum()),
    Avg_IMDb_Rating=('imdbAverageRating', 'mean'),
    Weighted_IMDb_Rating=('imdbAverageRating', lambda x: np.average(
        x.fillna(0), weights=catalog_df.loc[x.index, 'imdbNumVotes'].fillna(0))),
    Genre_Variety=('genres', lambda x: len(set(g for sublist in x.dropna().apply(eval) for g in sublist)))
).reset_index()

# Merge with pricing
summary['Platform'] = summary['platform'].map(platform_name_map)
summary = summary.merge(pricing_df, on='Platform', how='left')

# Value metrics
summary['Titles_per_Dollar'] = summary['Total_Titles'] / summary['Monthly Price']
summary['Originals_per_Dollar'] = summary['Total_Originals'] / summary['Monthly Price']
summary['Weighted_IMDb_per_Dollar'] = summary['Weighted_IMDb_Rating'] / summary['Monthly Price']

# Normalize
scaler = MinMaxScaler()
summary[['Norm_Titles_per_Dollar', 'Norm_Weighted_IMDb_per_Dollar', 'Norm_Originals_per_Dollar']] = scaler.fit_transform(
    summary[['Titles_per_Dollar', 'Weighted_IMDb_per_Dollar', 'Originals_per_Dollar']]
)
summary['Value_Index'] = (
    summary['Norm_Titles_per_Dollar'] +
    summary['Norm_Weighted_IMDb_per_Dollar'] +
    summary['Norm_Originals_per_Dollar']
)

# Melt for grouped bar chart
value_df = summary[['Platform', 'Titles_per_Dollar', 'Originals_per_Dollar', 'Weighted_IMDb_per_Dollar']]
value_df = value_df.melt(id_vars='Platform', var_name='Metric', value_name='Value')

# Custom colors for metrics
metric_colors = {
    'Titles_per_Dollar': '#3498db',
    'Originals_per_Dollar': '#2ecc71',
    'Weighted_IMDb_per_Dollar': '#9b59b6'
}

# Dash app
app = Dash(__name__)

app.layout = html.Div([
    html.H1("Streaming Platform Value for Money Analysis - 2025", style={'textAlign': 'center', 'color': 'white'}),

    dcc.Graph(
        figure=px.bar(
            summary.sort_values('Value_Index', ascending=True),
            x='Value_Index', y='Platform',
            orientation='h',
            color='Platform',
            color_discrete_map={
                'Netflix': '#E50914',
                'Amazon Prime Video': '#00A8E1',
                'Hulu': '#1CE783',
                'Max': '#5A2D82',
                'Apple TV+': '#A2AAAD'
            },
            title="Value Index by Platform"
        ).update_layout(paper_bgcolor='black', plot_bgcolor='black', font_color='white')
    ),

    dcc.Graph(
        figure=px.scatter(
            summary, x='Monthly Price', y='Value_Index',
            size='Total_Originals', color='Platform',
            color_discrete_map={
                'Netflix': '#E50914',
                'Amazon Prime Video': '#00A8E1',
                'Hulu': '#1CE783',
                'Max': '#5A2D82',
                'Apple TV+': '#A2AAAD'
            },
            hover_name='Platform',
            title="Price vs Value Index (Bubble = Number of Originals)"
        ).update_layout(paper_bgcolor='black', plot_bgcolor='black', font_color='white')
    ),

    dcc.Graph(
        figure=px.line_polar(
            value_df, r='Value', theta='Metric', color='Platform',
            line_close=True,
            color_discrete_map={
                'Netflix': '#E50914',
                'Amazon Prime Video': '#00A8E1',
                'Hulu': '#1CE783',
                'Max': '#5A2D82',
                'Apple TV+': '#A2AAAD'
            },
            title="Value Components per Dollar by Platform"
        ).update_layout(polar=dict(bgcolor='black'), paper_bgcolor='black', font_color='white')
    )
], style={'backgroundColor': 'black'})

if __name__ == '__main__':
    app.run(debug=True)



AttributeError: 'DataFrame' object has no attribute 'append'

In [27]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import plotly.express as px
from dash import Dash, html, dcc


# Pricing data manually entered
pricing_data = {
    'Platform': ['Netflix', 'Amazon Prime Video', 'Hulu', 'Max', 'Apple TV+'],
    'Plan': ['Standard (Ad-Free)', 'Prime with Video', 'No Ads', 'Ad-Free', 'Standard'],
    'Monthly Price': [17.99, 17.99, 18.99, 16.99, 9.99]
}
pricing_df = pd.DataFrame(pricing_data)

# Platform mapping
platform_name_map = {
    'netflix': 'Netflix',
    'prime': 'Amazon Prime Video',
    'hulu': 'Hulu',
    'hbo_max': 'Max',
    'apple_tv': 'Apple TV+'
}

# Summary table
summary = catalog_df.groupby('platform').agg(
    Total_Titles=('title', 'count'),
    Total_Originals=('is_original', lambda x: (x == 'Y').sum()),
    Avg_IMDb_Rating=('imdbAverageRating', 'mean'),
    Weighted_IMDb_Rating=('imdbAverageRating', lambda x: np.average(
        x.fillna(0), weights=catalog_df.loc[x.index, 'imdbNumVotes'].fillna(0))),
    Genre_Variety=('genres', lambda x: len(set(g for sublist in x.dropna().apply(eval) for g in sublist)))
).reset_index()

# Merge with pricing
summary['Platform'] = summary['platform'].map(platform_name_map)
summary = summary.merge(pricing_df, on='Platform', how='left')

# Value metrics
summary['Titles_per_Dollar'] = summary['Total_Titles'] / summary['Monthly Price']
summary['Originals_per_Dollar'] = summary['Total_Originals'] / summary['Monthly Price']
summary['Weighted_IMDb_per_Dollar'] = summary['Weighted_IMDb_Rating'] / summary['Monthly Price']

# Normalize
scaler = MinMaxScaler()
summary[['Norm_Titles_per_Dollar', 'Norm_Weighted_IMDb_per_Dollar', 'Norm_Originals_per_Dollar']] = scaler.fit_transform(
    summary[['Titles_per_Dollar', 'Weighted_IMDb_per_Dollar', 'Originals_per_Dollar']]
)
summary['Value_Index'] = (
    summary['Norm_Titles_per_Dollar'] +
    summary['Norm_Weighted_IMDb_per_Dollar'] +
    summary['Norm_Originals_per_Dollar']
)

# Melt for grouped bar chart
value_df = summary[['Platform', 'Titles_per_Dollar', 'Originals_per_Dollar', 'Weighted_IMDb_per_Dollar']]
value_df = value_df.melt(id_vars='Platform', var_name='Metric', value_name='Value')

# Custom colors for metrics
metric_colors = {
    'Titles_per_Dollar': '#3498db',
    'Originals_per_Dollar': '#2ecc71',
    'Weighted_IMDb_per_Dollar': '#9b59b6'
}

# Dash app
app = Dash(__name__)

app.layout = html.Div([
    html.H1("Streaming Platform Value for Money Analysis - 2025", style={'textAlign': 'center', 'color': 'white'}),

    dcc.Graph(
        figure=px.bar(
            summary.sort_values('Value_Index', ascending=True),
            x='Value_Index', y='Platform',
            orientation='h',
            color='Platform',
            color_discrete_map={
                'Netflix': '#E50914',
                'Amazon Prime Video': '#00A8E1',
                'Hulu': '#1CE783',
                'Max': '#5A2D82',
                'Apple TV+': '#A2AAAD'
            },
            title="Value Index by Platform"
        ).update_layout(paper_bgcolor='black', plot_bgcolor='black', font_color='white')
    ),

    dcc.Graph(
        figure=px.scatter(
            summary, x='Monthly Price', y='Value_Index',
            size='Total_Originals', color='Platform',
            color_discrete_map={
                'Netflix': '#E50914',
                'Amazon Prime Video': '#00A8E1',
                'Hulu': '#1CE783',
                'Max': '#5A2D82',
                'Apple TV+': '#A2AAAD'
            },
            hover_name='Platform',
            title="Price vs Value Index (Bubble = Number of Originals)"
        ).update_layout(paper_bgcolor='black', plot_bgcolor='black', font_color='white')
    ),

    dcc.Graph(
        figure=px.line_polar(
            value_df, r='Value', theta='Metric', color='Platform',
            line_close=True,
            color_discrete_map={
                'Netflix': '#E50914',
                'Amazon Prime Video': '#00A8E1',
                'Hulu': '#1CE783',
                'Max': '#5A2D82',
                'Apple TV+': '#A2AAAD'
            },
            title="Value Components per Dollar by Platform"
        ).update_layout(polar=dict(bgcolor='black'), paper_bgcolor='black', font_color='white')
    )
], style={'backgroundColor': 'black'})

if __name__ == '__main__':
    app.run(debug=True)


AttributeError: 'DataFrame' object has no attribute 'append'